# Validação de Integridade do Modelo

Nesta etapa, serão realizadas as validações dos relacionamentos entre as tabelas de dimensão (DIM) e a tabela fato (FATO), com o objetivo de identificar possíveis divergências e inconsistências nos dados, como registros órfãos e chaves sem correspondência. Após a validação, os dados estarão preparados para a próxima camada, Gold, onde serão estruturados para consumo analítico e correta apresentação das informações.

## 1. Importação das tabelas Silver

Nesta etapa, serão importadas as tabelas previamente tratadas e aprovadas na camada Silver, que serão utilizadas para realizar as validações de integridade e dos relacionamentos entre as dimensões e as tabelas fato.

In [1]:
# Importando bibliotecas
import pandas as pd
import numpy as np

# Importação das bases tratadas, da camada Staging:

from pathlib import Path

RAIZ    = Path.cwd().parent
STAGING = RAIZ / "data" / "Staging"
SILVER  = RAIZ / "data" / "Silver"

STAGING.mkdir(parents=True, exist_ok=True)
SILVER.mkdir(parents=True, exist_ok=True)

dim_cliente     = pd.read_csv(STAGING/'dim_clientes.csv', sep=';')
dim_assinatura  = pd.read_csv(STAGING/'dim_assinatura.csv', sep=';')
dim_interacoes  = pd.read_csv(STAGING/'dim_interacoes.csv', sep=';')
dim_planos      = pd.read_csv(STAGING/'dim_planos.csv', sep=';')
fato_pagamentos = pd.read_csv(STAGING/'fato_pagamentos.csv', sep=';')

## 2. Integridade das dimensões

Nesta etapa, os arquivos da camada Silver (Tabelas de Dimensão e Fato) são validados quanto à integridade de suas chaves primárias. A verificação assegura que os identificadores atendam aos seguintes critérios:

IDs não nulos: Garantem que não existam registros sem identificador.
IDs únicos: Evitam duplicações de registros e garantem a unicidade.
IDs válidos: Asseguram a consistência e conformidade com os domínios esperados.

Esse processo garante a qualidade e a confiabilidade dos dados para a próxima etapa do pipeline.

In [2]:
# 1. Criando uma função genérica de validação
def validar_chave_primaria(df, coluna_pk):
    return {
        'ID não nulo': df[coluna_pk].notna().all(),
        'ID único': df[coluna_pk].duplicated().sum() == 0,
        'ID válido': (df[coluna_pk] > 0).all()
    }

# Validando Integridade da tabela dim_clientes

## 1. Executando a função na tabela de clientes
resultado_cliente = validar_chave_primaria(dim_cliente, 'id_cliente')
print(f'Cliente: {resultado_cliente}')

## 2. Executando a função na tabela de assinatura
resultado_assinatura = validar_chave_primaria(dim_assinatura, 'id_assinatura')
print(f'Assinatura: {resultado_assinatura}')

## 3. Executando a função na tabela de planos
resultado_planos = validar_chave_primaria(dim_planos, 'id_plano')
print(f'Planos: {resultado_planos}')

## 4. Executando a função na tabela de interacoes
resultado_interacoes = validar_chave_primaria(dim_interacoes, 'id_interacao')
print(f'Interações: {resultado_interacoes}')

## 5. Executando a função na tabela de pagamentos
resultado_pagamentos = validar_chave_primaria(fato_pagamentos, 'id_pagamento')
print(f'Pagamento: {resultado_pagamentos}')


Cliente: {'ID não nulo': np.True_, 'ID único': np.True_, 'ID válido': np.True_}
Assinatura: {'ID não nulo': np.True_, 'ID único': np.True_, 'ID válido': np.True_}
Planos: {'ID não nulo': np.True_, 'ID único': np.True_, 'ID válido': np.True_}
Interações: {'ID não nulo': np.True_, 'ID único': np.True_, 'ID válido': np.True_}
Pagamento: {'ID não nulo': np.True_, 'ID único': np.True_, 'ID válido': np.True_}


## 3. Integridade referencial

Nesta etapa, será validado se as tabelas DIM possui uma relação com a Tabela Fato, a fim de seguir com as devidas tratativas.

In [3]:
# 1. Validação Integridade Referencial tabela (Fato):
def validar_relacionamento(df_filho, df_pai, coluna):
    """
    Verifica se todos os valores da coluna na tabela filha existem na tabela pai.
    Retorna True se o relacionamento estiver íntegro (sem órfãos) e False caso contrário.
    """
    return df_filho[coluna].isin(df_pai[coluna]).all()

# 1.2 Executando a validação relacional da tabela Pagamentos:
relacionamento_pagamento_assinatura = validar_relacionamento(fato_pagamentos, dim_assinatura, 'id_assinatura')
print(f"Relacionamento Pagamentos → Assinatura: {relacionamento_pagamento_assinatura}")

relacionamento_pagamento_cliente = validar_relacionamento(fato_pagamentos, dim_cliente, 'id_cliente')
print(f"Relacionamento Pagamentos → Cliente: {relacionamento_pagamento_cliente}")

# 1.3 Executando a validação relacional da tabela Interações:
relacionamento_interacao_cliente = validar_relacionamento(dim_interacoes, dim_cliente, 'id_cliente')
print(f"Relacionamento Interação → Cliente: {relacionamento_interacao_cliente}")

# 1.4 Executando a validação relacional da tabela Assinaturas:
relacionamento_assinatura_cliente = validar_relacionamento(dim_assinatura, dim_cliente, 'id_cliente')
print(f"Relacionamento Assinatura → Cliente: {relacionamento_assinatura_cliente}")

relacionamento_assinatura_plano = validar_relacionamento(dim_assinatura, dim_planos, 'id_plano')
print(f"Relacionamento Assinatura → Plano: {relacionamento_assinatura_plano}")

Relacionamento Pagamentos → Assinatura: False
Relacionamento Pagamentos → Cliente: False
Relacionamento Interação → Cliente: False
Relacionamento Assinatura → Cliente: False
Relacionamento Assinatura → Plano: False


In [4]:
# 3. Identificação dos registros órfãos:

def identificar_orfaos(df_filho, df_pai, coluna):
    """
    Retorna um DataFrame contendo apenas os registros da tabela filha 
    que não possuem correspondência na tabela pai (registros órfãos).
    """
    return df_filho[~df_filho[coluna].isin(df_pai[coluna])]

# 3.1 Executando para pagamento e assinatura
orfaos_pagamento_assinatura = identificar_orfaos(fato_pagamentos, dim_assinatura, 'id_assinatura')
print(f"Total de pagamentos órfãos por assinatura: {len(orfaos_pagamento_assinatura)}")

# 3.2 Executando para pagamento e cliente
orfaos_pagamento_cliente = identificar_orfaos(fato_pagamentos, dim_cliente,'id_cliente')
print(f"Total de pagamentos órfãos por cliente: {len(orfaos_pagamento_cliente)}")

# 3.3 Executando para Interacoes e cliente:
orfaos_interacao_cliente = identificar_orfaos(dim_interacoes, dim_cliente, 'id_cliente')
print(f"Total de interações órfãos por cliente: {len(orfaos_interacao_cliente)}")

# 3.4 Executando para Assinaturas e cliente e plano:
orfaos_assinatura_cliente = identificar_orfaos(dim_assinatura, dim_cliente, 'id_cliente')
print(f"Total de Assinaturas órfãos por cliente: {len(orfaos_assinatura_cliente)}")

orfaos_assinatura_plano = identificar_orfaos(dim_assinatura, dim_planos, 'id_plano')
print(f"Total de Assinaturas órfãos por plano: {len(orfaos_assinatura_plano)}")

Total de pagamentos órfãos por assinatura: 2
Total de pagamentos órfãos por cliente: 14
Total de interações órfãos por cliente: 5
Total de Assinaturas órfãos por cliente: 1
Total de Assinaturas órfãos por plano: 339


In [5]:
# 4. Verificação de quantas vezes cada Id órfão se repete
print(orfaos_pagamento_assinatura['id_assinatura'].value_counts())
print(orfaos_pagamento_cliente['id_cliente'].value_counts())
print(orfaos_interacao_cliente['id_cliente'].value_counts())
print(orfaos_assinatura_cliente['id_cliente'].value_counts())
print(orfaos_assinatura_plano['id_plano'].value_counts())

id_assinatura
77777    2
Name: count, dtype: int64
id_cliente
9999    14
Name: count, dtype: int64
id_cliente
9999    5
Name: count, dtype: int64
id_cliente
9999    1
Name: count, dtype: int64
id_plano
99    339
Name: count, dtype: int64


In [6]:
# 5. Validando se os registros órfãos se tratam de uma falha ou são válidos:
## órfãos cliente
dim_cliente[
    dim_cliente['id_cliente'] == 9999
]

## órfãos assinatura
dim_assinatura[
    dim_assinatura['id_assinatura'] == 77777
]

## órfãos plano
dim_planos[
    dim_planos['id_plano'] == 99
]

,id_plano,nome_plano,categoria,valor_mensal


## 3.1 Resultado da validação de integridade referencial

Foram identificados registros órfãos nas relações entre as tabelas DIM e FATO.

Os principais casos encontrados estão relacionados aos identificadores 9999, 77777 e 99, que possuem registros nas tabelas dependentes, mas não possuem correspondência nas respectivas tabelas de dimensão.

REGRA APLICADA:

Nesta etapa, os IDs órfãos serão substituídos pelo identificador -1, permitindo que esses registros sejam preservados e posteriormente identificados e analisados pelo time responsável. Dessa forma, será possível investigar a origem das inconsistências e atuar para evitar novas falhas de relacionamento entre os dados.

In [7]:
# Tratamento dos registros órfãos:

def tratar_orfaos(df_filho, df_pai, coluna, valor_padrao=-1):
    """
    Substitui IDs que não possuem correspondência na tabela pai
    pelo valor padrão definido.
    """
    df_filho[f'{coluna}_original'] = df_filho[coluna]
    df_filho.loc[
        ~df_filho[coluna].isin(df_pai[coluna]),
        coluna
    ] = valor_padrao
    
    return df_filho

fato_pagamentos = tratar_orfaos(
    fato_pagamentos,
    dim_assinatura,
    'id_assinatura'
)

fato_pagamentos = tratar_orfaos(
    fato_pagamentos,
    dim_cliente,
    'id_cliente'
)

dim_interacoes = tratar_orfaos(
    dim_interacoes,
    dim_cliente,
    'id_cliente'
)

dim_assinatura = tratar_orfaos(
    dim_assinatura,
    dim_cliente,
    'id_cliente'
)


dim_assinatura = tratar_orfaos(
    dim_assinatura,
    dim_planos,
    'id_plano'
)

In [8]:
# 2. Criação de Novos Registros, para que seja efetuado todo a conexão do relacionamento.

## 2.1 == Novo Registro na tabela plano.
novo_registro_planos = {
    'id_plano': -1,
    'nome_plano': 'Não identificado',
    'categoria': 'Não identificado',
    'valor_mensal': np.nan
}

## Inserindo registros na tabela.
dim_planos = pd.concat(
    [
        dim_planos,
        pd.DataFrame([novo_registro_planos])
    ],
    ignore_index=True
)

## 2.2 == Novo Registro na tabela assinatura.
novo_registro_assinatura = {
    'id_assinatura': -1,
    'id_cliente': -1,
    'id_plano': -1,
    'data_inicio': pd.NaT,
    'data_cancelamento': pd.NaT,
    'status': 'Não identificado',
    'reporte_data_cancelamento': pd.NaT,
    'id_cliente_original': -1,
    'id_plano_original': -1
}

## Inserindo registros na tabela.
dim_assinatura = pd.concat(
    [
        dim_assinatura,
        pd.DataFrame([novo_registro_assinatura])
    ],
    ignore_index=True
)

## 2.3 == Novo registro na tabela cliente
novo_registro_cliente = {
    'id_cliente': -1,
    'nome': 'Não identificado',
    'idade': np.nan,
    'cidade': 'Não identificado',
    'estado': 'NI',
    'data_cadastro': pd.NaT
}

## Inserindo registros na tabela.

dim_cliente = pd.concat(
    [
        dim_cliente,
        pd.DataFrame([novo_registro_cliente])
    ],
    ignore_index=True
)

# 2.4 A idade volta a ser decimal ao ler o CSV e ao inserir a linha -1.
dim_cliente['idade'] = dim_cliente['idade'].astype('Int64')

# 2.5 Validação:
print(dim_cliente['idade'].dtype)
print(dim_cliente['idade'].head(3).tolist())

Int64
[69, 27, 61]


## 3.2 Validação Pós Tratamento

Nesta etapa, será realizada a validação do tratamento de dados aplicado, após a identificação de registros órfãos. Aplicação da função: para validar se existe um relacionamento 100% entre as tabelas, caso retorne como True, significa que o tratamento está correto, caso retorno como False, sera necessário investigar.

In [9]:
## 1. Aplicação da função validar relacionamento
relacionamento_fato_dim_assinatura = validar_relacionamento(
    fato_pagamentos,
    dim_assinatura,
    'id_assinatura'
)

relacionamento_fato_dim_cliente = validar_relacionamento(
    fato_pagamentos,
    dim_cliente,
    'id_cliente'
)

relacionamento_interacao_dim_cliente = validar_relacionamento(
    dim_interacoes,
    dim_cliente,
    'id_cliente'
)

relacionamento_assinatura_dim_cliente = validar_relacionamento(
    dim_assinatura,
    dim_cliente,
    'id_cliente'
)

relacionamento_assinatura_dim_plano = validar_relacionamento(
    dim_assinatura,
    dim_planos,
    'id_plano'
)

## 2. Exibição de resultados pós tratamento:
integridade_aprovada = all([
    relacionamento_fato_dim_assinatura,
    relacionamento_fato_dim_cliente,
    relacionamento_interacao_dim_cliente,
    relacionamento_assinatura_dim_cliente,
    relacionamento_assinatura_dim_plano
])

## 3. Validação se os registros foram de fato inseridos nas tabelas:

print(fato_pagamentos[fato_pagamentos['id_assinatura'] == -1])
print(fato_pagamentos[fato_pagamentos['id_cliente'] == -1])
print(dim_interacoes[dim_interacoes['id_cliente'] == -1])
print(dim_assinatura[dim_assinatura['id_cliente'] == -1])
print(dim_assinatura[dim_assinatura['id_plano'] == -1])



       id_pagamento  id_assinatura  id_cliente data_pagamento   valor  \
12853         12854             -1        1064     2024-09-10  188.02   
13807         13808             -1        1033     2025-03-25  169.05   

      status_pagamento  valor_original  id_assinatura_original  \
12853         Atrasado          188.02                   77777   
13807             Pago          169.05                   77777   

       id_cliente_original  
12853                 1064  
13807                 1033  
       id_pagamento  id_assinatura  id_cliente data_pagamento   valor  \
1833           1834            460          -1     2025-02-21  185.82   
2924           2925            214          -1     2026-03-06   21.10   
3274           3275           2949          -1     2024-09-14   68.10   
3312           3313           1181          -1     2024-05-25   86.41   
3511           3512           2233          -1     2024-03-16  201.17   
5463           5464           1701          -1     2026-

      id_interacao  id_cliente data_interacao        tipo categoria  \
4432          4433          -1     2024-11-27  Reclamação  Cobrança   
4873          4874          -1     2025-12-27      Dúvida  Cobrança   
7503          7504          -1     2024-11-18    Sugestão   Produto   
7930          7931          -1     2025-06-26      Dúvida  Cobrança   
9987          9988          -1     2024-10-29     Suporte    Acesso   

      id_cliente_original  
4432                 9999  
4873                 9999  
7503                 9999  
7930                 9999  
9987                 9999  
      id_assinatura  id_cliente  id_plano data_inicio data_cancelamento  \
808             809          -1         4  2024-06-20               NaN   
3000             -1          -1        -1         NaT               NaT   

                status reporte_data_cancelamento  id_cliente_original  \
808              Ativo                        OK                 9999   
3000  Não identificado           

## 5. Relatório de integridade

Nesta etapa, será construido o relatório de integridade, a fim de validar se todos os relacionamentos estão corretos após todo o tratamento de dados.

In [10]:
resultado_integridade = {}

resultado_integridade['Pagamento → Assinatura'] = (
    validar_relacionamento(
        fato_pagamentos,
        dim_assinatura,
        'id_assinatura'
    )
)

resultado_integridade['Pagamento → Cliente'] = (
    validar_relacionamento(
        fato_pagamentos,
        dim_cliente,
        'id_cliente'
    )
)

resultado_integridade['Interação → Cliente'] = (
    validar_relacionamento(
        dim_interacoes,
        dim_cliente,
        'id_cliente'
    )
)

resultado_integridade['Assinatura → Cliente'] = (
    validar_relacionamento(
        dim_assinatura,
        dim_cliente,
        'id_cliente'
    )
)

resultado_integridade['Assinatura → Plano'] = (
    validar_relacionamento(
        dim_assinatura,
        dim_planos,
        'id_plano'
    )
)


## Transformando o relatório em um date frame:

df_integridade = pd.DataFrame(
    resultado_integridade.items(),
    columns=['Validação', 'Aprovado']
)



df_integridade

,Validação,Aprovado
0,Pagamento → Assinatura,True
1,Pagamento → Cliente,True
2,Interação → Cliente,True
3,Assinatura → Cliente,True
4,Assinatura → Plano,True


## 6. Quality Gate

Nesta etapa, será feita a análise dos resultados do relatório de integridade e haverá uma trava de segurança, onde em caso de qualquer relacionamento esteja divergente, os dados não passaram para a próxima etapa e será necessário investigar. Em caso do teste estar como Aprovado, os dados passam para a camada SILVER.

In [11]:
print(f"Integridade do modelo: {integridade_aprovada}")

Integridade do modelo: True


## 6.1 Modelo aprovado / reprovado

In [12]:
if integridade_aprovada:

    dim_cliente.to_csv(SILVER / 'dim_clientes.csv', index=False, sep=';', encoding='utf-8-sig')
    dim_assinatura.to_csv(SILVER / 'dim_assinatura.csv', index=False, sep=';', encoding='utf-8-sig')
    dim_planos.to_csv(SILVER / 'dim_planos.csv', index=False, sep=';', encoding='utf-8-sig')
    dim_interacoes.to_csv(SILVER / 'dim_interacoes.csv', index=False, sep=';', encoding='utf-8-sig')
    fato_pagamentos.to_csv(SILVER / 'fato_pagamentos.csv', index=False, sep=';', encoding='utf-8-sig')

    print("QUALITY GATE APROVADO")
    print("Sucesso: Todos os testes passaram. Os arquivos foram enviados para a camada Silver.")

else:

    print("QUALITY GATE REPROVADO")
    raise ValueError("Falha na integridade do modelo. A Silver não foi atualizada.")

QUALITY GATE APROVADO
Sucesso: Todos os testes passaram. Os arquivos foram enviados para a camada Silver.
